# Exploration des données BODACC

On va faire l'analyse dans un premier temps des fichiers BODACC

## 1. Chargement (année 2019)

In [1]:
import json
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", 120)
RAW = Path("../data/raw/bodacc")
df = pd.read_csv(RAW / "procedures_collectives_2019.csv.gz", sep=";", dtype=str)

In [2]:
df.shape

(173768, 13)

In [3]:
df.head()

,id,dateparution,typeavis_lib,numerodepartement,region_code,tribunal,commercant,ville,cp,registre,listepersonnes,jugement,parutionavisprecedent
0,A201900081030,2019-01-11,Avis initial,04,93,TRIBUNAL DE GRANDE INSTANCE DE DIGNE-LES-BAINS,"RICCO, Pascal",Manosque,04100,"394 335 467,394335467","{""personne"": {""typePersonne"": ""pp"", ""numeroImmatriculation"": {""numeroIdentification"": ""394 335 467"", ""codeRCS"": ""RCS...","{""famille"": ""Extrait de jugement"", ""nature"": ""Jugement prononçant la résolution du plan de redressement et la liquid...",NaN
1,A201900081377,2019-01-11,Avis initial,83,93,TRIBUNAL DE COMMERCE DE DRAGUIGNAN,"LE BOIS SAINT HUBERT, DOMAINE DE LA GREYTE",Rians,83560,"330 503 152,330503152,330 503 145,330503145","{""personne"": [{""typePersonne"": ""pm"", ""numeroImmatriculation"": {""numeroIdentification"": ""330 503 152"", ""codeRCS"": ""RC...","{""famille"": ""Jugement d'ouverture"", ""nature"": ""Jugement d'extension de liquidation judiciaire"", ""date"": ""2019-01-08""...",NaN
2,A201900081390,2019-01-11,Avis initial,84,93,TRIBUNAL DE COMMERCE D'AVIGNON,A L'HEURE DU GOUTER,Avignon,84000,"752 116 467,752116467","{""personne"": {""typePersonne"": ""pm"", ""numeroImmatriculation"": {""numeroIdentification"": ""752 116 467"", ""codeRCS"": ""RCS...","{""famille"": ""Avis de dépôt"", ""nature"": ""Liste des créances nées après le jugement d'ouverture d'une procédure de red...",NaN
3,A201900091569,2019-01-13,Avis initial,31,76,TRIBUNAL DE GRANDE INSTANCE DE TOULOUSE,"HADDAD, Joseph",Toulouse,31300,"791 920 614,791920614","{""personne"": {""typePersonne"": ""pp"", ""numeroImmatriculation"": {""numeroIdentification"": ""791 920 614"", ""codeRCS"": ""RCS...","{""famille"": ""Avis de dépôt"", ""nature"": ""Liste des créances nées après le jugement d'ouverture d'une procédure de red...",NaN
4,A201900091601,2019-01-13,Avis initial,36,24,TRIBUNAL DE COMMERCE DE CHÂTEAUROUX,GARAGE AMPERE,Buzançais,36500,"814 252 946,814252946","{""personne"": {""typePersonne"": ""pm"", ""numeroImmatriculation"": {""numeroIdentification"": ""814 252 946"", ""codeRCS"": ""RCS...","{""famille"": ""Avis de dépôt"", ""nature"": ""Liste des créances nées après le jugement d'ouverture d'une procédure de red...",NaN


## 2. Valeurs manquantes

In [4]:
df.isna().mean().sort_values(ascending=False)  # part de valeurs manquantes par colonne

parutionavisprecedent    0.993647
registre                 0.090408
cp                       0.002440
ville                    0.002440
jugement                 0.001416
commercant               0.000006
region_code              0.000000
typeavis_lib             0.000000
dateparution             0.000000
id                       0.000000
numerodepartement        0.000000
tribunal                 0.000000
listepersonnes           0.000000
dtype: float64

On voit que certaines données sont manquantes : parutionavisprecedent, registre, jugement. Et d'autres données sont des manques qu'on peut considérer comme négligeables : cp, ville, commerçant.

## 3. À quoi ressemble une annonce complète ?

In [5]:
def afficher_annonce(ligne):
    for col, val in ligne.items():
        if col in ("listepersonnes", "jugement") and isinstance(val, str):
            print(f"{col} :\n{json.dumps(json.loads(val), indent=2, ensure_ascii=False)}")
        else:
            print(f"{col} : {val}")

afficher_annonce(df.iloc[0])

id : A201900081030
dateparution : 2019-01-11
typeavis_lib : Avis initial
numerodepartement : 04
region_code : 93
tribunal : TRIBUNAL DE GRANDE INSTANCE DE DIGNE-LES-BAINS
commercant : RICCO, Pascal
ville : Manosque
cp : 04100
registre : 394 335 467,394335467
listepersonnes :
{
  "personne": {
    "typePersonne": "pp",
    "numeroImmatriculation": {
      "numeroIdentification": "394 335 467",
      "codeRCS": "RCS",
      "nomGreffeImmat": "de Digne-les-Bains"
    },
    "activite": "infirmier libéral",
    "adressePP": {
      "numeroVoie": "260",
      "typeVoie": "boulevard",
      "nomVoie": "du Temps-Perdu",
      "codePostal": "04100",
      "ville": "Manosque",
      "pays": "france"
    },
    "nom": "RICCO",
    "prenom": "Pascal"
  }
}
jugement :
{
  "famille": "Extrait de jugement",
  "nature": "Jugement prononçant la résolution du plan de redressement et la liquidation judiciaire",
  "date": "2018-12-13",
  "complementJugement": "Jugement de liquidation judiciaire. Représen

In [6]:
afficher_annonce(df.sample(1, random_state=42).iloc[0])

id : A201900812217
dateparution : 2019-04-25
typeavis_lib : Avis initial
numerodepartement : 73
region_code : 84
tribunal : TRIBUNAL DE COMMERCE DE CHAMBÉRY
commercant : ETRE
ville : Chambéry
cp : 73000
registre : 813 652 849,813652849
listepersonnes :
{
  "personne": {
    "typePersonne": "pm",
    "numeroImmatriculation": {
      "numeroIdentification": "813 652 849",
      "codeRCS": "RCS",
      "nomGreffeImmat": "Chambery"
    },
    "activite": "achat, vente de produits et d'articles de bien-être, produits d'hygiène, produits diététiques, plantes seules et en vrac, huiles essentielles, toutes activités liées à la Chromothérapie, toutes prestations liées au magnétisme.",
    "adresseSiegeSocial": {
      "numeroVoie": "120",
      "typeVoie": "rue",
      "nomVoie": "Croix d'Or",
      "codePostal": "73000",
      "ville": "Chambéry",
      "pays": "france"
    },
    "denomination": "ETRE",
    "formeJuridique": "Société à responsabilité limitée à associé unique"
  }
}
jugement :

### Observations

- Une annonce correspond à **un jugement** (un événement juridique) qui concerne **une ou plusieurs personnes**.
- Le premier exemple est une **personne physique** (un infirmier libéral, avec nom, prénom et adresse personnelle). Le second est une **personne morale** (une SARL, avec dénomination, forme juridique et siège social). La structure de `listepersonnes` change selon le cas.
- **La date du jugement n'est pas la date de parution** : le premier jugement date du 13/12/2018 mais il est publié le 11/01/2019. L'année du fichier est celle de la **parution**.
- Le second exemple est une **clôture** : la fin d'une procédure ouverte plus tôt, pas une nouvelle défaillance.
- Le texte libre `complementJugement` contient des informations utiles mais non structurées, comme la date de cessation des paiements ou le mandataire.

## 4. Types d'avis et avis précédent

In [7]:
df["typeavis_lib"].value_counts()

typeavis_lib
Avis initial         172664
Avis rectificatif       858
Avis d’annulation       246
Name: count, dtype: int64

In [8]:
pd.crosstab(df["typeavis_lib"], df["parutionavisprecedent"].notna())

parutionavisprecedent,False,True
typeavis_lib,,
Avis d’annulation,0,246
Avis initial,172664,0
Avis rectificatif,0,858


In [9]:
df.loc[df["parutionavisprecedent"].notna(), ["typeavis_lib", "parutionavisprecedent"]].head()

,typeavis_lib,parutionavisprecedent
128,Avis rectificatif,"{""nomPublication"": ""BODACC A"", ""numeroParution"": ""20180234"", ""dateParution"": ""2018-12-07"", ""numeroAnnonce"": ""3326""}"
926,Avis rectificatif,"{""nomPublication"": ""BODACC A"", ""numeroParution"": ""20180225"", ""dateParution"": ""2018-11-25"", ""numeroAnnonce"": ""2775""}"
941,Avis rectificatif,"{""nomPublication"": ""BODACC A"", ""numeroParution"": ""20180224"", ""dateParution"": ""2018-11-23"", ""numeroAnnonce"": ""3209""}"
1115,Avis d’annulation,"{""nomPublication"": ""BODACC A"", ""numeroParution"": ""20180148"", ""dateParution"": ""2018-08-07"", ""numeroAnnonce"": ""2456""}"
1340,Avis d’annulation,"{""nomPublication"": ""BODACC A"", ""numeroParution"": ""20180246"", ""dateParution"": ""2018-12-26"", ""numeroAnnonce"": ""2430""}"


In [10]:
# Retrouver l'annonce d'origine visée par un rectificatif / une annulation
prec = df["parutionavisprecedent"].dropna().map(json.loads)
id_origine = prec.map(lambda d: "A" + d["numeroParution"] + d["numeroAnnonce"])

ids_2018_2019 = pd.concat([pd.read_csv(RAW / f"procedures_collectives_{a}.csv.gz", sep=";", dtype=str, usecols=["id"])["id"]
                           for a in (2018, 2019)])
print(f"{id_origine.isin(set(ids_2018_2019)).sum()} annonces d'origine retrouvées sur {len(id_origine)}")
prec.map(lambda d: d["dateParution"][:4]).value_counts()  # année de l'annonce d'origine

1076 annonces d'origine retrouvées sur 1104


parutionavisprecedent
2019    933
2018    152
2017      8
2012      4
2011      2
2014      1
3019      1
2016      1
2010      1
2013      1
Name: count, dtype: int64

### Observations

- 99,4 % des annonces sont des **avis initiaux**. On compte 858 **rectificatifs** (correction d'une annonce déjà publiée) et 246 **annulations** (une annonce publiée par erreur).
- `parutionavisprecedent` est rempli **exactement** pour les rectificatifs et les annulations, et jamais pour les avis initiaux. Ce n'est donc pas une donnée manquante, c'est un champ réservé à ces deux cas.
- Ce champ permet de **retrouver l'annonce d'origine** : son `id` vaut `"A" + numeroParution + numeroAnnonce`. Sur 2018-2019, 1 076 des 1 104 annonces visées sont retrouvées. Les autres visent des annonces plus anciennes, plus une date aberrante (« 3019 »).
- Les annulations n'ont **pas de jugement** : ce sont exactement les 246 annonces où `jugement` est vide (voir section 5).

### Décision (Luka)

- **Rectificatifs : on les garde.** Ils mettent à jour le jugement, donc le rectificatif **remplace** l'annonce d'origine.
- **Annulations : on ne les garde pas en tant qu'annonces.** Elles servent seulement à **retirer les jugements annulés** (publiés par erreur) : l'annonce d'origine visée est supprimée, l'annulation aussi.
- *Mise en œuvre à l'étape de nettoyage, sur toutes les années (l'annonce d'origine peut être dans un fichier antérieur).*

## 5. Le champ `jugement` : que s'est-il passé ?

### Lexique : comprendre les jugements

Une procédure collective se déroule **en plusieurs étapes**, et chaque étape fait l'objet d'une annonce au BODACC. Le champ `famille` indique l'étape, le champ `nature` le jugement précis.

```
                 ┌──────────────── SAUVEGARDE (pas en cessation des paiements)
                 │                    └─> plan de sauvegarde ─> fin de procédure
 OUVERTURE ──────┼──────────────── REDRESSEMENT JUDICIAIRE (RJ) (en cessation des paiements, sauvetage possible)
                 │                    ├─> plan de redressement (l'entreprise continue)
                 │                    ├─> plan de cession (l'entreprise est reprise par un tiers)
                 │                    └─> CONVERSION en liquidation ──┐
                 └──────────────── LIQUIDATION JUDICIAIRE (LJ) directe ─┴─> vente des actifs ─> CLÔTURE
```

#### Les 8 familles

| Famille | À quelle étape ? | Exemples de natures |
|---|---|---|
| **Jugement d'ouverture** | **Début** de la procédure. C'est le moment où l'entreprise entre en procédure collective. | Ouverture de sauvegarde, de redressement judiciaire, de liquidation judiciaire |
| **Jugement prononçant** | **Changement de procédure** en cours de route | Conversion d'un RJ en LJ, résolution d'un plan de sauvegarde suivie d'une liquidation |
| **Extrait de jugement** | **Décisions intermédiaires** sur une procédure déjà ouverte | Arrêt d'un plan, modification d'un plan, interdiction de gérer, faillite personnelle |
| **Avis de dépôt** | **Formalités** du mandataire, sans décision du tribunal | Dépôt de l'état des créances, projet de répartition |
| **Jugement de clôture** | **Fin** de la procédure | Clôture pour insuffisance d'actif, clôture pour extinction du passif |
| **Arrêt de la Cour d'Appel** | **Recours** : la cour d'appel confirme ou annule (« infirme ») une décision | — |
| **Rétractation sur tierce opposition** | **Annulation** d'un jugement à la demande d'un tiers qui le conteste | Rétractation d'un jugement d'ouverture |
| **Loi de 1967** | **Ancien régime** des faillites (« liquidation des biens »), avant la loi de 1985 : ce sont des procédures très anciennes qui se clôturent encore | Clôture de la liquidation des biens |

#### Les termes à connaître

| Terme | Signification |
|---|---|
| **Cessation des paiements** | L'entreprise ne peut plus payer ses dettes exigibles avec ce dont elle dispose. C'est la **frontière** entre la sauvegarde (avant) et le redressement ou la liquidation (après). |
| **Sauvegarde** | Procédure **préventive** demandée par le dirigeant **avant** la cessation des paiements, pour réorganiser l'entreprise à l'abri des créanciers. |
| **Sauvegarde financière accélérée** | Variante rapide de la sauvegarde, réservée aux grandes entreprises qui négocient avec leurs créanciers financiers. Elle est très rare. |
| **Redressement judiciaire (RJ)** | Entreprise **en cessation des paiements** mais qu'on pense **sauvable**. Une période d'observation détermine son sort. |
| **Liquidation judiciaire (LJ)** | Entreprise **en cessation des paiements** et **impossible à sauver**. L'activité s'arrête et les biens sont vendus. Une LJ « directe » est ouverte d'emblée, sans passer par un RJ. |
| **Conversion** | Passage d'une procédure à une plus grave (sauvegarde → RJ, ou RJ → LJ) quand le sauvetage échoue. L'entreprise **était déjà** en procédure. |
| **Extension** | La procédure d'une entreprise est **étendue** à une autre personne, souvent une société liée ou le dirigeant, quand leurs patrimoines sont confondus. |
| **Plan de sauvegarde / de redressement / de continuation** | Accord validé par le tribunal : l'entreprise **continue** et rembourse ses dettes par étapes, jusqu'à 10 ans. « Continuation » est l'ancien nom du plan de redressement. |
| **Plan de cession** | L'entreprise (ou une partie) est **vendue à un repreneur**, qui reprend l'activité et souvent une partie des salariés. |
| **Résolution du plan** | Le plan n'est **pas respecté**. Il est annulé, ce qui entraîne en général une liquidation (« résolution du plan… **et** la liquidation judiciaire »). |
| **Désignation des organes** | Nomination des acteurs de la procédure : administrateur, mandataire judiciaire. |
| **État des créances** | Liste de toutes les dettes de l'entreprise, vérifiées par le mandataire. Les créanciers peuvent la contester. |
| **Créances nées après le jugement d'ouverture** | Dettes **contractées pendant** la procédure. Elles sont traitées à part. |
| **État de collocation / projet de répartition** | Le classement des créanciers et la répartition de l'argent obtenu par la vente des actifs. |
| **Clôture pour insuffisance d'actif** | Fin de la liquidation : il n'y a **plus rien à vendre** et les dettes restantes ne seront pas payées. C'est le cas le plus fréquent. |
| **Clôture pour extinction du passif** | Fin de la procédure parce que **toutes les dettes ont été payées**. C'est rare. |
| **Interdiction de gérer / faillite personnelle** | **Sanctions contre le dirigeant** en cas de faute de gestion. Elles visent la personne, pas l'entreprise. |
| **Reprise des poursuites individuelles** | Après la clôture, le tribunal autorise exceptionnellement les créanciers à poursuivre de nouveau le débiteur (en cas de fraude, par exemple). |
| **Loi de 1985** | Ancien régime (avant la réforme de 2005) : des procédures ouvertes il y a longtemps et encore en cours. |

In [11]:
jug_json = df["jugement"].dropna().map(json.loads)
jug = pd.json_normalize(jug_json.tolist())
jug.index = jug_json.index  # réaligne sur df
jug.columns.tolist()

['famille', 'nature', 'date', 'complementJugement', 'type']

In [12]:
jug["famille"].value_counts()

famille
Jugement de clôture                   53881
Jugement d'ouverture                  49901
Avis de dépôt                         37873
Extrait de jugement                   16344
Jugement prononçant                   14940
Arrêt de la Cour d'Appel                525
Rétractation sur tierce opposition       44
Loi de 1967                              14
Name: count, dtype: int64

In [13]:
# Natures les plus fréquentes, par famille (plus de 100 annonces)
(jug.groupby(["famille", "nature"]).size()
    .loc[lambda s: s > 100]
    .sort_values(ascending=False)
    .to_frame("nb_annonces"))

,,nb_annonces
famille,nature,
Jugement de clôture,Jugement de clôture pour insuffisance d'actif,52363
Jugement d'ouverture,Jugement d'ouverture de liquidation judiciaire,31902
Avis de dépôt,Dépôt de l'état des créances,24866
Jugement d'ouverture,Jugement d'ouverture d'une procédure de redressement judiciaire,16241
Jugement prononçant,Jugement de conversion en liquidation judiciaire,11556
Avis de dépôt,Liste des créances nées après le jugement d'ouverture d'une procédure de liquidation judiciaire,8159
Extrait de jugement,Jugement de plan de redressement,3963
Jugement prononçant,Autre jugement prononçant,2791
Extrait de jugement,Jugement d'interdiction de gérer,2537


In [14]:
# Zoom sur les ouvertures
jug.loc[jug["famille"] == "Jugement d'ouverture", "nature"].value_counts()

nature
Jugement d'ouverture de liquidation judiciaire                             31902
Jugement d'ouverture d'une procédure de redressement judiciaire            16241
Jugement d'ouverture d'une procédure de sauvegarde                          1012
Autre jugement d'ouverture                                                   337
Jugement d'extension de liquidation judiciaire                               254
Jugement d'extension d'une procédure de redressement judiciaire               95
Jugement de désignation des organes de la procédure                           40
Jugement d'extension d'une procédure de sauvegarde                            18
Jugement d'ouverture d'une procédure de sauvegarde financière accélérée        2
Name: count, dtype: int64

In [15]:
# Délai entre la date du jugement et sa publication au BODACC (en jours)
delai = (pd.to_datetime(df["dateparution"]) - pd.to_datetime(jug["date"], errors="coerce")).dt.days
delai.describe()

count    173374.000000
mean         17.431276
std        1191.513126
min     -365225.000000
25%           8.000000
50%          10.000000
75%          16.000000
max       32516.000000
dtype: float64

In [16]:
# Annonces sans jugement
df.loc[df["jugement"].isna(), ["dateparution", "typeavis_lib", "commercant", "tribunal"]].head()

,dateparution,typeavis_lib,commercant,tribunal
1115,2019-01-23,Avis d’annulation,LE FOURNIL DE CATHERINE S.A.R.L.,TRIBUNAL DE GRANDE INSTANCE DE ST-DENIS
1340,2019-01-03,Avis d’annulation,CHOLSTER,TRIBUNAL DE COMMERCE DE VERSAILLES
1341,2019-01-03,Avis d’annulation,LE CLOS LECLERC,TRIBUNAL DE COMMERCE DE VERSAILLES
1342,2019-01-03,Avis d’annulation,SCI RUE DE LA REPUBLIQUE,TRIBUNAL DE COMMERCE DE VERSAILLES
1813,2019-01-23,Avis d’annulation,E.A.R.L. DE COAT CESSOU,TRIBUNAL DE GRANDE INSTANCE DE PÉRIGUEUX


In [17]:
# Les annonces sans jugement sont-elles les annulations ?
pd.crosstab(df["typeavis_lib"], df["jugement"].isna())

jugement,False,True
typeavis_lib,,
Avis d’annulation,0,246
Avis initial,172664,0
Avis rectificatif,858,0


### Observations

- Les **4 familles principales** en 2019 sont les clôtures (53 881), les ouvertures (49 901), les avis de dépôt (37 873) et les extraits de jugement (16 344), suivies des jugements prononçant (14 940).
- Parmi les **ouvertures** : 31 902 **liquidations directes**, 16 241 **redressements**, 1 012 **sauvegardes**. On trouve aussi des extensions (367 au total), 337 « autres » et 40 désignations d'organes, qui ne sont pas de nouvelles procédures.
- Des événements de **passage en liquidation** existent **hors de la famille « ouverture »** :
  - 11 556 **conversions RJ → LJ** et 202 conversions sauvegarde → LJ (famille « Jugement prononçant ») ;
  - 2 418 **résolutions de plan de redressement suivies d'une liquidation** (famille « Extrait de jugement »).

  Ces entreprises étaient **déjà en procédure** auparavant.
- Une grande partie des annonces (clôtures, dépôts) concerne la **fin** ou la **gestion** de procédures ouvertes les années précédentes. Compter toutes les annonces reviendrait à compter plusieurs fois les mêmes entreprises, avec un décalage dans le temps.
- **Délai jugement → parution** : la médiane est de 10 jours et 75 % des annonces paraissent en moins de 16 jours. Mais le minimum (−365 225 jours) et le maximum (+32 516 jours) montrent des **dates de jugement aberrantes**, des fautes de saisie probables, à traiter au nettoyage.
- Les 246 annonces **sans jugement** sont exactement les 246 **avis d'annulation**.

### Décision (Luka)

Il existe **plusieurs types de défaillance**, qui contribuent tous à l'analyse avec des gravités différentes. On crée une variable **« type d'événement »** au lieu d'un simple défaillant oui ou non :

| Catégorie | Événements | Rôle dans l'analyse |
|---|---|---|
| **Signal précoce** | Ouverture de sauvegarde | Difficulté **sans** cessation des paiements, ce n'est pas une défaillance |
| **Entrée en défaillance** | Ouverture de redressement judiciaire, ouverture de liquidation judiciaire directe | La **défaillance** au sens de la Banque de France, comptée **une fois** par entreprise |
| **Aggravation** | Conversion (sauvegarde → RJ, RJ → LJ), résolution de plan suivie d'une liquidation | L'entreprise était déjà en procédure. C'est un indicateur de **gravité**, pas une nouvelle défaillance |
| **Issue** | Plan de sauvegarde ou de redressement, plan de cession, clôture | Le **devenir** de l'entreprise |

## 6. Le champ `listepersonnes` : qui est concerné ?

In [18]:
# 'personne' est un dict (1 personne) ou une liste (plusieurs personnes)
lp = df["listepersonnes"].map(lambda s: json.loads(s)["personne"])
nb_personnes = lp.map(lambda p: len(p) if isinstance(p, list) else 1)
nb_personnes.value_counts()

listepersonnes
1      172137
2        1279
3         256
4          61
6          11
5          10
7           4
8           3
9           3
13          1
30          1
351         1
10          1
Name: count, dtype: int64

In [19]:
# On regarde la première personne de chaque annonce
pers = pd.json_normalize(lp.map(lambda p: p[0] if isinstance(p, list) else p).tolist())
pers.index = df.index
pers.notna().mean().sort_values(ascending=False)  # taux de remplissage des sous-champs

typePersonne                                  1.000000
activite                                      0.986263
numeroImmatriculation.codeRCS                 0.885503
numeroImmatriculation.numeroIdentification    0.885503
numeroImmatriculation.nomGreffeImmat          0.885503
denomination                                  0.808900
formeJuridique                                0.807761
adresseSiegeSocial.pays                       0.807324
adresseSiegeSocial.ville                      0.806869
adresseSiegeSocial.codePostal                 0.806869
adresseSiegeSocial.nomVoie                    0.754742
adresseSiegeSocial.typeVoie                   0.709831
adresseSiegeSocial.numeroVoie                 0.664104
adressePP.pays                                0.190731
adressePP.codePostal                          0.190685
adressePP.ville                               0.190685
prenom                                        0.187572
nom                                           0.187572
adresseSie

In [20]:
pers["typePersonne"].value_counts()

typePersonne
pm    140561
pp     33207
Name: count, dtype: int64

In [21]:
pers["formeJuridique"].value_counts().head(15)

formeJuridique
Société à responsabilité limitée                     58070
Société par actions simplifiée                       22387
Société à Responsabilité Limitée                     14267
Société à responsabilité limitée à associé unique     8603
Société par Actions Simplifiée                        8049
Société par actions simplifiée à associé unique       7301
SARL                                                  4336
SCI                                                   2846
association                                           1957
SAS                                                   1133
Société anonyme                                        923
EARL                                                   897
Association                                            820
SASU                                                   580
Société en nom collectif                               475
Name: count, dtype: int64

In [22]:
pers["activite"].sample(10, random_state=0)

73852     production, distribution et vente d'électricité à partir d'énergies renouvelables, intermédiaire dans le commerce d'...
101702                                                                                                               non précisée
157705                                                                                                Restauration traditionnelle
55020     -courtage d'assurances-courtage bancaire - contentieux et recouvrement de créances - opérations immobilières - opéra...
135675    acquisition et exploitation directe ou en location-gérance de tous fonds de plomberie, chauffage, sanitaire, de tout...
66311                                                                                                          Débits de boissons
161883                                                                                          boucherie, charcuterie, Traîteur.
23568                                                                                     

### Observations

- **81 % de personnes morales** (`pm` : sociétés, associations…) et **19 % de personnes physiques** (`pp` : entrepreneurs individuels, professions libérales, agriculteurs…).
- 99 % des annonces concernent **une seule personne**. 1 631 annonces en concernent plusieurs, jusqu'à 351 dans une seule annonce, probablement une procédure étendue à tout un groupe.
- Les sous-champs dépendent du type de personne : `denomination`, `formeJuridique` et `adresseSiegeSocial` pour les personnes morales (~81 %), `nom`, `prenom` et `adressePP` pour les personnes physiques (~19 %).
- `inscriptionRM` (6,6 %) signale une inscription au **Répertoire des Métiers** : ce sont les **artisans**.
- **`formeJuridique` n'est pas harmonisé** : la SARL apparaît sous au moins 3 écritures (« Société à responsabilité limitée », « Société à Responsabilité Limitée », « SARL »). Même chose pour SAS, SASU et association. La **catégorie juridique SIRENE** (codée) sera plus fiable.
- **`activite` est un texte libre**, parfois très long, parfois « non précisée ». On ne pourra pas en tirer un secteur fiable : le **code NAF de SIRENE** sera nécessaire.
- ⚠️ Ces champs contiennent des **données personnelles** (nom, prénom, adresse des personnes physiques) qui ne doivent pas être diffusées.

## 7. Les annonces sans SIREN (`registre` vide)

In [23]:
sans_siren = df["registre"].isna()
pd.crosstab(pers["typePersonne"], sans_siren, margins=True)

registre,False,True,All
typePersonne,,,
pm,136391,4170,140561
pp,21667,11540,33207
All,158058,15710,173768


In [24]:
# Quels sous-champs sont remplis pour les annonces sans SIREN ?
pers.loc[sans_siren].notna().mean().loc[lambda s: s > 0].sort_values(ascending=False)

typePersonne                            1.000000
nonInscrit                              1.000000
activite                                0.911649
adressePP.pays                          0.732718
adressePP.ville                         0.732336
adressePP.codePostal                    0.732336
prenom                                  0.729981
nom                                     0.729981
adressePP.nomVoie                       0.642903
adressePP.typeVoie                      0.580649
adressePP.numeroVoie                    0.577976
denomination                            0.265436
formeJuridique                          0.264799
adresseSiegeSocial.pays                 0.264163
adresseSiegeSocial.ville                0.263463
adresseSiegeSocial.codePostal           0.263463
adresseSiegeSocial.nomVoie              0.234437
adresseSiegeSocial.typeVoie             0.216168
adresseSiegeSocial.numeroVoie           0.191216
adressePP.complGeographique             0.164545
adresseSiegeSocial.c

In [25]:
cols = [c for c in pers.columns if c in ('typePersonne', 'denomination', 'nom', 'activite', 'formeJuridique')
        or c.startswith('numeroImmatriculation')]
pers.loc[sans_siren, cols].sample(10, random_state=0)

,typePersonne,activite,nom,numeroImmatriculation.numeroIdentification,numeroImmatriculation.codeRCS,numeroImmatriculation.nomGreffeImmat,denomination,formeJuridique
41632,pm,non précisée,NaN,NaN,NaN,NaN,SYNDICAT DES COPROPRIETAIRES,syndicat de copropriété
151718,pp,NaN,MONSIEUR PINTO DA SILVA,NaN,NaN,NaN,NaN,NaN
110331,pp,non précisé,CLAPIER,NaN,NaN,NaN,NaN,NaN
87707,pp,moniteur auto-école,VILAS,NaN,NaN,NaN,NaN,NaN
119598,pp,non précisée,CORBIN,NaN,NaN,NaN,NaN,NaN
111250,pp,NaN,PINOT,NaN,NaN,NaN,NaN,NaN
119178,pp,NaN,BALLARO,NaN,NaN,NaN,NaN,NaN
44878,pm,Non précisée,NaN,NaN,NaN,NaN,ASSOCIATION SOLIDAIRE PEUPLE WE “SPW”,Association
16584,pp,non précisée,PETIT,NaN,NaN,NaN,NaN,NaN
116311,pm,non précisée,NaN,NaN,NaN,NaN,EARL LE SOLEIL DES LORIOTS,EARL


In [26]:
display(pers["nonInscrit"].value_counts(dropna=False))
pd.crosstab(pers["nonInscrit"].notna(), sans_siren, rownames=["nonInscrit renseigné"], colnames=["sans SIREN"])

nonInscrit
NaN                153872
RCS non inscrit     19896
Name: count, dtype: int64

sans SIREN,False,True
nonInscrit renseigné,,
False,153872,0
True,4186,15710


### Observations

- **15 710 annonces sans SIREN** (9 %) : 11 540 personnes physiques (73 %) et 4 170 personnes morales.
- **Toutes** ont le sous-champ `nonInscrit = "RCS non inscrit"`. Elles ne sont **pas inscrites au Registre du Commerce et des Sociétés**, d'où l'absence de numéro d'immatriculation dans l'annonce. Ce sont typiquement des professions libérales, des agriculteurs, des associations ou des personnes physiques non commerçantes.
- Autre fait : 4 186 annonces ont un SIREN **et** la mention « RCS non inscrit ». Le marqueur ne veut donc pas toujours dire « pas de SIREN ».
- Ces annonces gardent la **date**, le **département**, le **tribunal**, le type de personne et souvent une **activité** en texte libre.
- Sans SIREN, elles ne peuvent **pas être reliées** à SIRENE (secteur NAF, taille, âge) ni aux ratios. On ne peut pas non plus **dédoublonner** de façon fiable les annonces d'une même entreprise.

### Décision (Luka)

On **garde** les annonces sans SIREN. Elles ne permettront pas d'identifier les secteurs touchés, mais elles contribuent aux **comptages globaux** (dans le temps, par territoire), par exemple pour mesurer l'effet d'événements ou de décisions gouvernementales.

Nature de ces annonces : une entreprise doit exister juridiquement pour faire l'objet d'une procédure collective, donc ce ne sont **pas des entreprises en création**. Ce sont des entités **non inscrites au RCS** : **professions libérales**, **agriculteurs**, **associations** ou **personnes physiques non commerçantes**.

Limites : pas de dédoublonnage possible (on compte des **annonces**, pas des entreprises), et c'est une population différente des sociétés, à **ne pas mélanger** sans le signaler.

## 8. Le champ `registre` : combien d'annonces par entreprise ?

In [27]:
# Format : 'SIREN avec espaces,SIREN sans espaces' répété s'il y a plusieurs entreprises
df["registre"].dropna().sample(5, random_state=0).tolist()

['450 039 698,450039698',
 '798 625 497,798625497',
 '791 308 745,791308745',
 '830 647 392,830647392',
 '797 819 000,797819000']

In [28]:
# On extrait les SIREN à 9 chiffres (sans espaces) de chaque annonce
sirens = df["registre"].str.findall(r"\b\d{9}\b")
sirens.str.len().value_counts().sort_index()  # nombre de SIREN par annonce

registre
1.0      157644
2.0         353
3.0          25
4.0          10
5.0          12
6.0           1
7.0           9
10.0          1
12.0          1
21.0          1
351.0         1
Name: count, dtype: int64

In [29]:
# Nombre d'annonces par SIREN en 2019
annonces_par_siren = sirens.explode().dropna().value_counts()
display(annonces_par_siren.describe())
annonces_par_siren.value_counts().sort_index().head(10)

count    121579.000000
mean          1.307693
std           0.576170
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          22.000000
Name: count, dtype: float64

count
1     90023
2     26574
3      4286
4       588
5        86
6        12
7         3
8         3
9         2
20        1
Name: count, dtype: int64

### Observations

- `registre` contient le SIREN **deux fois** (avec et sans espaces), et autant de fois qu'il y a d'entreprises dans l'annonce. Il faut donc **extraire** les SIREN à 9 chiffres.
- 157 644 annonces portent 1 seul SIREN. Quelques-unes en portent plusieurs, jusqu'à 351, ce qui correspond aux annonces à plusieurs personnes vues en section 6.
- En 2019, on compte **121 579 SIREN distincts** pour environ 158 000 annonces avec SIREN. 74 % des entreprises n'ont **qu'une** annonce dans l'année, 22 % en ont **deux**, et le maximum est de 22.
- Une même entreprise apparaît donc plusieurs fois, à chaque étape de sa procédure. **L'unité d'analyse devra être l'entreprise (SIREN) ou la procédure, pas l'annonce.** Une procédure peut aussi s'étaler sur plusieurs années, donc sur plusieurs fichiers.

## 9. Vue d'ensemble 2008-2026 : volume d'annonces par année et par famille de jugement

In [30]:
frames = []
for f in sorted(RAW.glob("procedures_collectives_*.csv.gz")):
    d = pd.read_csv(f, sep=";", dtype=str, usecols=["dateparution", "typeavis_lib", "jugement"])
    # extraction de la famille par regex : beaucoup plus rapide que json.loads sur 3,4 M de lignes
    d["famille"] = d["jugement"].str.extract(r'"famille": "([^"]*)"', expand=False)
    frames.append(d.drop(columns="jugement"))

toutes = pd.concat(frames, ignore_index=True)
toutes["annee"] = toutes["dateparution"].str[:4].astype(int)
toutes.shape

(3365280, 4)

In [31]:
pd.crosstab(toutes["annee"], toutes["famille"], margins=True)

famille,ArrÃªt de la Cour d'Appel,Arrêt de la Cour d'Appel,Avis de dÃ©pÃ´t,Avis de dépôt,Extrait de jugement,Jugement d'ouverture,Jugement de clÃ´ture,Jugement de clôture,Jugement prononÃ§ant,Jugement prononçant,Loi de 1967,Rétractation sur tierce opposition,All
annee,,,,,,,,,,,,,
2008,0,344,0,37326,13115,55014,0,49877,0,11298,179,104,167257
2009,0,457,0,40392,14592,62821,0,51275,0,16354,113,85,186089
2010,0,552,0,46693,16663,60572,0,59790,0,16756,118,106,201250
2011,0,486,0,48055,16458,58724,0,58345,0,16292,110,103,198573
2012,0,429,0,46815,16708,58771,0,57194,0,15764,57,78,195816
2013,0,427,0,45082,16440,61474,0,58513,0,15459,58,76,197529
2014,0,466,0,45702,17101,59746,0,59074,0,17239,46,63,199437
2015,0,482,0,46048,17442,60283,0,64066,0,17847,21,69,206258
2016,1,998,8,45089,17924,56464,8,62175,8,17651,21,77,200424


In [32]:
# Lignes dont le texte contient un encodage cassé (« Ã »), par année
encodage = []
for f in sorted(RAW.glob("procedures_collectives_*.csv.gz")):
    d = pd.read_csv(f, sep=";", dtype=str)
    casse = d.apply(lambda c: c.str.contains("Ã", na=False)).any(axis=1)
    encodage.append({"annee": int(f.stem[-8:-4]), "lignes": len(d), "encodage_casse": casse.sum()})
pd.DataFrame(encodage).set_index("annee")

,lignes,encodage_casse
annee,,
2008,167257,0
2009,186158,3
2010,201378,17
2011,198705,45
2012,195917,21
2013,197622,19
2014,199535,14
2015,206358,1
2016,200542,1114


### Observations

- **3 365 280 annonces** de 2008 à 2026. **2026 est incomplète** : les données s'arrêtent au 31/10/2026.
- ⚠️ **Problème d'encodage dans la source** : sur 2016-2018, une partie des textes a des accents « cassés » (`ArrÃªt` au lieu de `Arrêt`, `dÃ©pÃ´t` au lieu de `dépôt`). Une même famille apparaît alors dans **deux colonnes** du tableau. Ça touche 1 114 lignes en 2016, 7 916 en 2017 et **17 018 en 2018** (environ 10 %), contre 1 ligne en 2015 et en 2019. Le défaut est **dans les données publiées par la DILA**, pas dans le téléchargement (vérifié via l'API). **À corriger au nettoyage**, sinon les comptages 2016-2018 seront faux.
- Le tableau donne une première idée des **volumes** par famille et par année. Il ne faut pas l'interpréter avant d'avoir corrigé l'encodage et défini ce qu'on compte.

## Récapitulatif des décisions pour le nettoyage

| # | Sujet | Décision (Luka) |
|---|---|---|
| 1 | **Rectificatifs** | Gardés : ils **remplacent** l'annonce d'origine (jugement mis à jour) |
| 2 | **Annulations** | Elles servent à **retirer** l'annonce d'origine. L'annulation elle-même n'est pas gardée |
| 3 | **Rétractations sur tierce opposition, arrêts de cour d'appel infirmant une décision** | Traités **comme des annulations** : on retire le jugement visé quand on peut l'identifier |
| 4 | **Type de défaillance** | Variable « type d'événement » : signal précoce (sauvegarde), entrée en défaillance (ouverture de RJ ou de LJ directe), aggravation (conversions, résolutions de plan), issue (plans, cession, clôture) |
| 5 | **Extensions de procédure** | L'entité visée **entre en procédure** : c'est une entrée en défaillance pour elle |
| 6 | **Défaillances multiples** | Une entreprise peut défaillir **plusieurs fois** : on compte chaque défaillance, mais **une seule entreprise**. Il faut vérifier qu'on sait identifier qu'il s'agit de la même entreprise |
| 7 | **Annonces sans SIREN** | Gardées pour les comptages globaux (temps, territoire). Ce sont des entités non inscrites au RCS (professions libérales, agriculteurs, associations, personnes physiques non commerçantes) |
| 8 | **Annonces à plusieurs entreprises** | **Une ligne par SIREN**, avec un indicateur « annonce collective » |
| 9 | **Date de l'événement** | **Date du jugement**. On prend la date de parution si la date du jugement est aberrante (seuil à fixer) |
| 10 | **Période** | 2008 → octobre 2026. **2026 est exclue** des comparaisons annuelles, ou comparée à janvier-octobre des autres années. La rupture Covid 2020-2021 est toujours signalée |
| 11 | **Encodage 2016-2018** | Réparé au nettoyage (correction technique) |